In [4]:

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import unicodedata
import re
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "c6aaad32-bd37-4408-b7e5-4dcd52f68e45",
   "metadata": {},
   "outputs": [
    {
     "ename": "NameError",
     "evalue": "name 'nombres_ine' is not defined",
     "output_type": "error",
     "traceback": [
      "\u001b[31m---------------------------------------------------------------------------\u001b[39m",
      "\u001b[31mNameError\u001b[39m                                 Traceback (most recent call last)",
      "\u001b[36mCell\u001b[39m\u001b[36m \u001b[39m\u001b[32mIn[2]\u001b[39m\u001b[32m, line 37\u001b[39m\n\u001b[32m     33\u001b[39m \u001b[38;5;66;03m# Eliminar registros que no corresponden a comunidades autónomas\u001b[39;00m\n\u001b[32m     34\u001b[39m df_year = df_year[df_year[\u001b[33m\"CCAA\"\u001b[39m] != \u001b[33m\"Extranjero\"\u001b[39m]\n\u001b[32m     35\u001b[39m \n\u001b[32m     36\u001b[39m \u001b[38;5;66;03m# Adaptar los nombres\u001b[39;00m\n\u001b[32m---> \u001b[39m\u001b[32m37\u001b[39m df_year[\u001b[33m\"CCAA\"\u001b[39m] = df_year[\u001b[33m\"CCAA\"\u001b[39m].replace(nombres_ine)\n\u001b[32m     38\u001b[39m \n\u001b[32m     39\u001b[39m \u001b[38;5;66;03m# Calcular el promedio por comunidad autónoma\u001b[39;00m\n\u001b[32m     40\u001b[39m df_year = (\n",
      "\u001b[31mNameError\u001b[39m: name 'nombres_ine' is not defined"
     ]
    }
   ],
   "source": [
    "\n",
    "import pandas as pd\n",
    "import geopandas as gpd\n",
    "import matplotlib.pyplot as plt\n",
    "import unicodedata\n",
    "import re\n",
    "\n",
    "# ---------------------------------------\n",
    "# 1. CARGAR LOS DATOS DEL INE\n",
    "# ---------------------------------------\n",
    "\n",
    "df = pd.read_csv(\n",
    "    \"datos_ine_suicidio_ccaa_desde2001.csv\",\n",
    "    encoding=\"latin1\",\n",
    "    sep=None,\n",
    "    engine=\"python\"\n",
    ")\n",
    "\n",
    "# Seleccionar un año\n",
    "#year = 2012\n",
    "\n",
    "#df_year = df[df[\"Periodo\"] == year].copy()\n",
    "\n",
    "#_________________________________________________________________________\n",
    "# Calcular el promedio de suicidios por CCAA\n",
    "df_year = df.copy()\n",
    "\n",
    "# Eliminar códigos numéricos al principio del nombre\n",
    "df_year[\"CCAA\"] = (\n",
    "    df_year[\"Comunidades y Ciudades Autónomas\"]\n",
    "    .str.replace(r\"^\\d+\\s+\", \"\", regex=True)\n",
    ")\n",
    "\n",
    "# Eliminar registros que no corresponden a comunidades autónomas\n",
    "df_year = df_year[df_year[\"CCAA\"] != \"Extranjero\"]\n",
    "\n",
    "# Adaptar los nombres\n",
    "df_year[\"CCAA\"] = df_year[\"CCAA\"].replace(nombres_ine)\n",
    "\n",
    "# Calcular el promedio por comunidad autónoma\n",
    "df_year = (\n",
    "    df_year.groupby(\"CCAA\", as_index=False)[\"Total\"]\n",
    "    .mean()\n",
    ")\n",
    "\n",
    "# Redondear a dos decimales\n",
    "df_year[\"Total\"] = df_year[\"Total\"].round(2)\n",
    "#_________________________________________________________________________\n",
    "\n",
    "# Eliminar códigos numéricos al principio del nombre\n",
    "df_year[\"CCAA\"] = (\n",
    "    df_year[\"Comunidades y Ciudades Autónomas\"]\n",
    "    .str.replace(r\"^\\d+\\s+\", \"\", regex=True)\n",
    ")\n",
    "\n",
    "# Eliminar registros que no corresponden a comunidades autónomas\n",
    "df_year = df_year[df_year[\"CCAA\"] != \"Extranjero\"]\n",
    "\n",
    "\n",
    "# ---------------------------------------\n",
    "# 2. NORMALIZAR LOS NOMBRES\n",
    "# ---------------------------------------\n",
    "\n",
    "# Adaptar los nombres del INE a los nombres geográficos\n",
    "nombres_ine = {\n",
    "    \"Asturias, Principado de\": \"Principado de Asturias\",\n",
    "    \"Balears, Illes\": \"Illes Balears\",\n",
    "    \"Castilla - La Mancha\": \"Castilla-La Mancha\",\n",
    "    \"Comunitat Valenciana\": \"Comunidad Valenciana\",\n",
    "    \"Madrid, Comunidad de\": \"Comunidad de Madrid\",\n",
    "    \"Murcia, Región de\": \"Región de Murcia\",\n",
    "    \"Navarra, Comunidad Foral de\": \"Comunidad Foral de Navarra\",\n",
    "    \"Rioja, La\": \"La Rioja\"\n",
    "}\n",
    "\n",
    "df_year[\"CCAA\"] = df_year[\"CCAA\"].replace(nombres_ine)\n",
    "\n",
    "\n",
    "# Función para comparar nombres ignorando acentos,\n",
    "# mayúsculas y espacios\n",
    "def normalizar(nombre):\n",
    "    nombre = unicodedata.normalize(\"NFKD\", str(nombre))\n",
    "    nombre = \"\".join(\n",
    "        c for c in nombre\n",
    "        if not unicodedata.combining(c)\n",
    "    )\n",
    "    nombre = nombre.lower().strip()\n",
    "    nombre = re.sub(r\"[-_,]\", \" \", nombre)\n",
    "    return \" \".join(nombre.split())\n",
    "\n",
    "\n",
    "df_year[\"join_name\"] = df_year[\"CCAA\"].apply(normalizar)\n",
    "\n",
    "\n",
    "# ---------------------------------------\n",
    "# 3. CARGAR LOS LÍMITES GEOGRÁFICOS\n",
    "# ---------------------------------------\n",
    "\n",
    "url = (\n",
    "    \"https://mapas.fomento.gob.es/arcgis/rest/services/\"\n",
    "    \"SIU/ENTIDADES_TERRITORIALES_EGRN/MapServer/1/query\"\n",
    "    \"?where=1%3D1&outFields=*&outSR=4326&f=geojson\"\n",
    ")\n",
    "\n",
    "gdf = gpd.read_file(url)\n",
    "\n",
    "# Comprobar las columnas del archivo geográfico\n",
    "print(gdf.columns)\n",
    "\n",
    "\n",
    "# ---------------------------------------\n",
    "# 4. PREPARAR LA UNIÓN DE LOS DATOS\n",
    "# ---------------------------------------\n",
    "\n",
    "# El servicio geográfico utiliza NAMEUNIT\n",
    "gdf[\"join_name\"] = gdf[\"NAMEUNIT\"].apply(normalizar)\n",
    "\n",
    "# Unir geometrías y datos del INE\n",
    "mapa = gdf.merge(\n",
    "    df_year[[\"join_name\", \"Total\"]],\n",
    "    on=\"join_name\",\n",
    "    how=\"left\"\n",
    ")\n",
    "\n",
    "# Comprobar si hay comunidades sin datos\n",
    "print(mapa[[\"NAMEUNIT\", \"Total\"]])\n",
    "\n",
    "\n",
    "# ---------------------------------------\n",
    "# 5. REPRESENTAR EL MAPA\n",
    "# ---------------------------------------\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(12, 10))\n",
    "\n",
    "mapa.plot(\n",
    "    column=\"Total\",\n",
    "    cmap=\"Reds\",\n",
    "    linewidth=0.8,\n",
    "    edgecolor=\"black\",\n",
    "    legend=True,\n",
    "    legend_kwds={\n",
    "        \"label\": \"Número de suicidios\",\n",
    "        \"orientation\": \"horizontal\",\n",
    "        \"shrink\": 0.7\n",
    "    },\n",
    "    missing_kwds={\n",
    "        \"color\": \"lightgrey\",\n",
    "        \"label\": \"Sin datos\"\n",
    "    },\n",
    "    ax=ax\n",
    ")\n",
    "\n",
    "ax.set_title(\n",
    "    f\"Suicidios por comunidad autónoma en España ({year})\",\n",
    "    fontsize=16,\n",
    "    pad=20\n",
    ")\n",
    "\n",
    "ax.axis(\"off\")\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.16"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

# ---------------------------------------
# 1. NORMALIZAR LOS NOMBRES DEL INE
# ---------------------------------------

nombres_ine = {
    "Asturias, Principado de": "Principado de Asturias",
    "Balears, Illes": "Illes Balears",
    "Castilla - La Mancha": "Castilla-La Mancha",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Madrid, Comunidad de": "Comunidad de Madrid",
    "Murcia, Región de": "Región de Murcia",
    "Navarra, Comunidad Foral de": "Comunidad Foral de Navarra",
    "Rioja, La": "La Rioja"
}

# ---------------------------------------
# 2. CARGAR LOS DATOS DEL INE
# ---------------------------------------

df = pd.read_csv(
    "datos_ine_suicidio_ccaa_desde2001.csv",
    encoding="latin1",
    sep=None,
    engine="python"
)

# Seleccionar todos los años
df_year = df.copy()

# Si quieres seleccionar un año concreto:
# year = 2012
# df_year = df[df["Periodo"] == year].copy()

# ---------------------------------------
# 3. PREPARAR LOS DATOS
# ---------------------------------------

df_year["CCAA"] = (
    df_year["Comunidades y Ciudades Autónomas"]
    .str.replace(r"^\d+\s+", "", regex=True)
)

# Eliminar registros que no corresponden a CCAA
df_year = df_year[df_year["CCAA"] != "Extranjero"].copy()

# Adaptar los nombres
df_year["CCAA"] = df_year["CCAA"].replace(nombres_ine)

# Calcular el promedio por comunidad autónoma
df_year = (
    df_year.groupby("CCAA", as_index=False)["Total"]
    .mean()
)

df_year["Total"] = df_year["Total"].round(2)

# ---------------------------------------
# 4. NORMALIZAR LOS NOMBRES
# ---------------------------------------

def normalizar(nombre):
    nombre = unicodedata.normalize("NFKD", str(nombre))
    nombre = "".join(
        c for c in nombre
        if not unicodedata.combining(c)
    )
    nombre = nombre.lower().strip()
    nombre = re.sub(r"[-_,]", " ", nombre)
    return " ".join(nombre.split())

df_year["join_name"] = df_year["CCAA"].apply(normalizar)

# ---------------------------------------
# 5. CARGAR LOS LÍMITES GEOGRÁFICOS
# ---------------------------------------

url = (
    "https://mapas.fomento.gob.es/arcgis/rest/services/"
    "SIU/ENTIDADES_TERRITORIALES_EGRN/MapServer/1/query"
    "?where=1%3D1&outFields=*&outSR=4326&f=geojson"
)

gdf = gpd.read_file(url)

print(gdf.columns)

# ---------------------------------------
# 6. PREPARAR LA UNIÓN DE LOS DATOS
# ---------------------------------------

gdf["join_name"] = gdf["NAMEUNIT"].apply(normalizar)

mapa = gdf.merge(
    df_year[["join_name", "Total"]],
    on="join_name",
    how="left"
)

# Comprobar comunidades sin datos
print(mapa[["NAMEUNIT", "Total"]])

# ---------------------------------------
# 7. REPRESENTAR EL MAPA
# ---------------------------------------

fig, ax = plt.subplots(figsize=(12, 10))

mapa.plot(
    column="Total",
    cmap="Reds",
    linewidth=0.8,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Número medio de suicidios",
        "orientation": "horizontal",
        "shrink": 0.7
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "Sin datos"
    },
    ax=ax
)

ax.set_title(
    "Promedio de suicidios por comunidad autónoma en España",
    fontsize=16,
    pad=20
)

ax.axis("off")

plt.tight_layout()
plt.show()


IndentationError: unexpected indent (794285217.py, line 6)